<a href="https://colab.research.google.com/github/MNMARQUES/Python/blob/main/Projeto03/Python_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install flask-sqlalchemy

from flask import Flask, request, jsonify
from flask_sqlalchemy import SQLAlchemy

# Inicialização da aplicação
app = Flask(__name__)

# Configuração do banco de dados SQLite
app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///usuarios.db"
app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False

db = SQLAlchemy(app)


# Model da entidade Usuario
class Usuario(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    nome = db.Column(db.String(100), nullable=False)
    telefone = db.Column(db.String(20), nullable=False)
    endereco = db.Column(db.String(200), nullable=False)
    cpf = db.Column(db.String(14), unique=True, nullable=False)

    def to_dict(self):
        return {
            "id": self.id,
            "nome": self.nome,
            "telefone": self.telefone,
            "endereco": self.endereco,
            "cpf": self.cpf
        }


# Criação das tabelas
with app.app_context():
    db.create_all()


# 1. Criar usuário - CREATE
@app.route("/usuarios", methods=["POST"])
def criar_usuario():
    dados = request.get_json(silent=True)

    if not dados:
        return jsonify({"erro": "Envie os dados em formato JSON."}), 400

    campos = ["nome", "telefone", "endereco", "cpf"]

    if any(not isinstance(dados.get(campo), str)
           or not dados[campo].strip() for campo in campos):
        return jsonify({
            "erro": "Informe nome, telefone, endereco e cpf."
        }), 400

    if Usuario.query.filter_by(cpf=dados["cpf"].strip()).first():
        return jsonify({"erro": "CPF já cadastrado."}), 409

    usuario = Usuario(
        nome=dados["nome"].strip(),
        telefone=dados["telefone"].strip(),
        endereco=dados["endereco"].strip(),
        cpf=dados["cpf"].strip()
    )

    db.session.add(usuario)
    db.session.commit()

    return jsonify(usuario.to_dict()), 201


# 2. Consultar todos os usuários - READ
@app.route("/usuarios", methods=["GET"])
def listar_usuarios():
    usuarios = Usuario.query.order_by(Usuario.id).all()
    return jsonify([usuario.to_dict() for usuario in usuarios]), 200


# 3. Consultar um usuário pelo ID - READ
@app.route("/usuarios/<int:id>", methods=["GET"])
def consultar_usuario(id):
    usuario = db.session.get(Usuario, id)

    if usuario is None:
        return jsonify({"erro": "Usuário não encontrado."}), 404

    return jsonify(usuario.to_dict()), 200


# 4. Atualizar usuário - UPDATE
@app.route("/usuarios/<int:id>", methods=["PUT"])
def atualizar_usuario(id):
    usuario = db.session.get(Usuario, id)

    if usuario is None:
        return jsonify({"erro": "Usuário não encontrado."}), 404

    dados = request.get_json(silent=True)

    if not isinstance(dados, dict) or not dados:
        return jsonify({"erro": "Envie os dados em formato JSON."}), 400

    campos = ["nome", "telefone", "endereco", "cpf"]

    if any(campo not in campos for campo in dados):
        return jsonify({"erro": "Campo inválido."}), 400

    for campo, valor in dados.items():
        if not isinstance(valor, str) or not valor.strip():
            return jsonify({
                "erro": f"O campo {campo} deve ser um texto não vazio."
            }), 400

    novo_cpf = dados.get("cpf", usuario.cpf).strip()
    duplicado = Usuario.query.filter(
        Usuario.cpf == novo_cpf,
        Usuario.id != id
    ).first()

    if duplicado:
        return jsonify({"erro": "CPF já cadastrado por outro usuário."}), 409

    for campo, valor in dados.items():
        setattr(usuario, campo, valor.strip())

    db.session.commit()

    return jsonify(usuario.to_dict()), 200


# 5. Excluir usuário - DELETE
@app.route("/usuarios/<int:id>", methods=["DELETE"])
def deletar_usuario(id):
    usuario = db.session.get(Usuario, id)

    if usuario is None:
        return jsonify({"erro": "Usuário não encontrado."}), 404

    db.session.delete(usuario)
    db.session.commit()

    return jsonify({"mensagem": "Usuário excluído com sucesso."}), 200


# Rota inicial
@app.route("/", methods=["GET"])
def inicio():
    return jsonify({
        "mensagem": "API de usuários funcionando!",
        "endpoints": [
            "POST /usuarios",
            "GET /usuarios",
            "GET /usuarios/<id>",
            "PUT /usuarios/<id>",
            "DELETE /usuarios/<id>"
        ]
    }), 200


print("Aplicação Flask e endpoints criados com sucesso!")

Aplicação Flask e endpoints criados com sucesso!


In [3]:

# Cliente de testes da API
cliente = app.test_client()

# CREATE: cadastrar um usuário fictício
novo_usuario = {
    "nome": "João da Silva",
    "telefone": "21999990000",
    "endereco": "Rua das Flores, 100 - Rio de Janeiro",
    "cpf": "111.222.333-44"
}

resposta = cliente.post("/usuarios", json=novo_usuario)

print("1. CREATE - Criar usuário")
print("Status:", resposta.status_code)
print("Resposta:", resposta.get_json())

# Guardar o ID criado para as próximas operações
usuario_id = resposta.get_json()["id"]

# READ: consultar todos os usuários
resposta = cliente.get("/usuarios")

print("\n2. READ - Listar usuários")
print("Status:", resposta.status_code)
print("Resposta:", resposta.get_json())

# READ: consultar usuário específico
resposta = cliente.get(f"/usuarios/{usuario_id}")

print("\n3. READ - Consultar usuário por ID")
print("Status:", resposta.status_code)
print("Resposta:", resposta.get_json())

# UPDATE: atualizar telefone e endereço
dados_atualizados = {
    "telefone": "21988887777",
    "endereco": "Avenida Brasil, 200 - Rio de Janeiro"
}

resposta = cliente.put(
    f"/usuarios/{usuario_id}",
    json=dados_atualizados
)

print("\n4. UPDATE - Atualizar usuário")
print("Status:", resposta.status_code)
print("Resposta:", resposta.get_json())

# DELETE: excluir o usuário
resposta = cliente.delete(f"/usuarios/{usuario_id}")

print("\n5. DELETE - Excluir usuário")
print("Status:", resposta.status_code)
print("Resposta:", resposta.get_json())

# Confirmar que o usuário foi removido
resposta = cliente.get(f"/usuarios/{usuario_id}")

print("\n6. Validação da exclusão")
print("Status:", resposta.status_code)
print("Resposta:", resposta.get_json())


1. CREATE - Criar usuário
Status: 201
Resposta: {'cpf': '111.222.333-44', 'endereco': 'Rua das Flores, 100 - Rio de Janeiro', 'id': 1, 'nome': 'João da Silva', 'telefone': '21999990000'}

2. READ - Listar usuários
Status: 200
Resposta: [{'cpf': '111.222.333-44', 'endereco': 'Rua das Flores, 100 - Rio de Janeiro', 'id': 1, 'nome': 'João da Silva', 'telefone': '21999990000'}]

3. READ - Consultar usuário por ID
Status: 200
Resposta: {'cpf': '111.222.333-44', 'endereco': 'Rua das Flores, 100 - Rio de Janeiro', 'id': 1, 'nome': 'João da Silva', 'telefone': '21999990000'}

4. UPDATE - Atualizar usuário
Status: 200
Resposta: {'cpf': '111.222.333-44', 'endereco': 'Avenida Brasil, 200 - Rio de Janeiro', 'id': 1, 'nome': 'João da Silva', 'telefone': '21988887777'}

5. DELETE - Excluir usuário
Status: 200
Resposta: {'mensagem': 'Usuário excluído com sucesso.'}

6. Validação da exclusão
Status: 404
Resposta: {'erro': 'Usuário não encontrado.'}
